# 06 - Prompting & Inference: Temperature, Top-K, Top-P (OPTIONAL)

---

> **This notebook is OPTIONAL.** It covers decoding strategies for text generation using only PyTorch and random logits -- **no API calls or large models required.**

## Learning Objectives

By the end of this notebook, you will be able to:

- Explain the **decoder-only Transformer** concept (GPT-style next-token prediction)
- Implement **temperature scaling** and understand its effect on output distributions
- Implement **top-k sampling**: restrict to the top $k$ most probable tokens
- Implement **top-p (nucleus) sampling**: restrict to the smallest set with cumulative probability $\geq p$
- Demonstrate the effects of each strategy on random logit distributions
- Combine strategies for controlled text generation

## Prerequisites

- Understanding of softmax and probability distributions
- Completed Notebooks 01-02 (attention, Transformer architecture)
- No external libraries beyond PyTorch and NumPy

## Table of Contents

1. [Decoder-Only Transformers (GPT-Style)](#1-decoder-only-transformers-gpt-style)
2. [Temperature Scaling](#2-temperature-scaling)
3. [Code: Implement Temperature Scaling](#3-code-implement-temperature-scaling)
4. [Top-K Sampling](#4-top-k-sampling)
5. [Code: Implement Top-K Filtering](#5-code-implement-top-k-filtering)
6. [Top-P (Nucleus) Sampling](#6-top-p-nucleus-sampling)
7. [Code: Implement Top-P Filtering](#7-code-implement-top-p-filtering)
8. [Combining Strategies](#8-combining-strategies)
9. [Code: Demonstrate Effects with Random Logits](#9-code-demonstrate-effects-with-random-logits)
10. [Exercise: Experiment with Parameter Combinations](#10-exercise-experiment-with-parameter-combinations)
11. [Common Mistakes & Debugging Tips](#11-common-mistakes--debugging-tips)

In [ ]:
import sys
sys.path.insert(0, "../..")
from src.utils.seed import set_global_seed

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

set_global_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

print(f"PyTorch version: {torch.__version__}")

---
## 1. Decoder-Only Transformers (GPT-Style)

**Decoder-only Transformers** (GPT, GPT-2, GPT-3, LLaMA, etc.) are trained for **next-token prediction**:

Given a sequence of tokens $[t_1, t_2, \ldots, t_n]$, predict $t_{n+1}$.

**Architecture:**
- Stack of Transformer decoder blocks (causal self-attention + FFN)
- No encoder, no cross-attention
- Causal mask: each token only attends to previous tokens (no peeking at the future)

**Generation process (autoregressive):**
1. Feed the prompt: `["The", "cat"]`
2. Model outputs logits for position 3 (a score for every token in the vocabulary)
3. **Sample** the next token from the logit distribution -> e.g., `"sat"`
4. Append to sequence: `["The", "cat", "sat"]`
5. Repeat until a stop condition

**The key question:** How do we convert logits to a token selection? That is where temperature, top-k, and top-p come in.

### Greedy vs Sampling

- **Greedy decoding:** Always pick the highest-probability token. Deterministic but can be repetitive and dull.
- **Sampling:** Randomly sample from the probability distribution. More diverse but can be incoherent.
- **Controlled sampling:** Use temperature, top-k, top-p to balance diversity and coherence.

---
## 2. Temperature Scaling

**Temperature** controls the "sharpness" of the probability distribution.

Given logits $z = [z_1, z_2, \ldots, z_V]$ (one per vocabulary token), the temperature-scaled probability is:

$$p_i = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$

where $T$ is the temperature:

| Temperature | Effect | Use Case |
|:-:|:-:|:-:|
| $T \to 0$ | Becomes greedy (argmax) | Factual, deterministic output |
| $T = 1$ | Standard softmax | Default behavior |
| $T > 1$ | Flatter distribution (more random) | Creative, diverse output |
| $T < 1$ | Sharper distribution (more confident) | Focused, high-confidence output |

**Intuition:** Temperature divides the logits before softmax. Smaller $T$ amplifies differences between logits (making the top token even more dominant); larger $T$ smooths differences (making all tokens more equally likely).

---
## 3. Code: Implement Temperature Scaling

In [ ]:
def apply_temperature(logits, temperature):
    """
    Apply temperature scaling to logits.
    
    Args:
        logits: (vocab_size,) tensor of raw logits
        temperature: float > 0. Lower = more confident, higher = more uniform.
    Returns:
        Probability distribution (vocab_size,) tensor
    """
    if temperature <= 0:
        raise ValueError("Temperature must be positive")
    scaled_logits = logits / temperature
    return F.softmax(scaled_logits, dim=-1)


# Demonstrate with example logits
set_global_seed(42)

# Simulate logits for a vocabulary of 10 tokens
vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran", "big", "red", "hat"]
logits = torch.tensor([2.0, 1.5, 1.0, 0.5, 0.0, -0.5, -1.0, -1.5, -2.0, -2.5])

temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]

print(f"{'Token':<8} {'Logit':>6}", end="")
for t in temperatures:
    print(f" {'T='+str(t):>8}", end="")
print()
print("-" * 60)

for i, token in enumerate(vocab):
    print(f"{token:<8} {logits[i]:>6.1f}", end="")
    for t in temperatures:
        probs = apply_temperature(logits, t)
        print(f" {probs[i]:>8.4f}", end="")
    print()

In [ ]:
# Visualize temperature effects
fig, axes = plt.subplots(1, len(temperatures), figsize=(4 * len(temperatures), 4))

for ax, t in zip(axes, temperatures):
    probs = apply_temperature(logits, t).numpy()
    bars = ax.bar(range(len(vocab)), probs, color='steelblue', alpha=0.8)
    ax.set_xticks(range(len(vocab)))
    ax.set_xticklabels(vocab, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 1.0)
    ax.set_title(f'T = {t}', fontsize=12)
    ax.set_ylabel('Probability')
    
    # Highlight the max
    max_idx = np.argmax(probs)
    bars[max_idx].set_color('tomato')
    
    # Entropy
    entropy = -np.sum(probs * np.log(probs + 1e-10))
    ax.text(0.95, 0.95, f'H={entropy:.2f}', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Effect of Temperature on Probability Distribution', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Low temperature (T=0.1): Nearly all probability mass on 'the' (greedy-like).")
print("High temperature (T=5.0): Almost uniform distribution (very random).")
print("H = entropy: higher entropy means more randomness.")

---
## 4. Top-K Sampling

**Top-k sampling** restricts the next token to only the $k$ most probable tokens, then renormalizes:

1. Compute probabilities from logits (with optional temperature)
2. Keep only the top $k$ tokens by probability
3. Set all other probabilities to 0
4. Renormalize so probabilities sum to 1
5. Sample from this filtered distribution

**Effect:** Prevents sampling very unlikely tokens (which can cause incoherent text), while still allowing diversity among the top candidates.

**Limitation:** A fixed $k$ may be too restrictive when the distribution is flat (many reasonable options) or too loose when the distribution is peaked (one obvious choice).

---
## 5. Code: Implement Top-K Filtering

In [ ]:
def top_k_filtering(logits, k):
    """
    Filter logits to keep only the top-k values.
    All other logits are set to -inf (zero probability after softmax).
    
    Args:
        logits: (vocab_size,) tensor of raw logits
        k: number of top tokens to keep
    Returns:
        Filtered logits (vocab_size,) tensor
    """
    if k <= 0:
        raise ValueError("k must be positive")
    if k >= logits.size(-1):
        return logits  # keep all
    
    # Find the k-th largest value
    top_k_values, _ = torch.topk(logits, k)
    threshold = top_k_values[-1]  # smallest value in top-k
    
    # Mask out everything below the threshold
    filtered = logits.clone()
    filtered[logits < threshold] = float('-inf')
    
    return filtered


# Demonstrate
print("Original logits:")
for i, (token, logit) in enumerate(zip(vocab, logits.tolist())):
    print(f"  {token}: {logit:.1f}")

print()
for k in [3, 5, 8]:
    filtered = top_k_filtering(logits, k)
    probs = F.softmax(filtered, dim=-1)
    print(f"Top-{k} probabilities:")
    for i, (token, p) in enumerate(zip(vocab, probs.tolist())):
        if p > 0:
            print(f"  {token}: {p:.4f}")
    print()

In [ ]:
# Visualize top-k effects
k_values = [2, 3, 5, 10]

fig, axes = plt.subplots(1, len(k_values), figsize=(4 * len(k_values), 4))

for ax, k in zip(axes, k_values):
    filtered = top_k_filtering(logits, k)
    probs = F.softmax(filtered, dim=-1).numpy()
    
    colors = ['steelblue' if p > 0 else 'lightgray' for p in probs]
    ax.bar(range(len(vocab)), probs, color=colors, alpha=0.8)
    ax.set_xticks(range(len(vocab)))
    ax.set_xticklabels(vocab, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 0.8)
    ax.set_title(f'Top-K = {k}', fontsize=12)
    ax.set_ylabel('Probability')
    
    n_active = sum(1 for p in probs if p > 0)
    ax.text(0.95, 0.95, f'{n_active} tokens', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Effect of Top-K on Token Selection', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Top-P (Nucleus) Sampling

**Top-p sampling** (also called **nucleus sampling**) dynamically selects the **smallest set** of tokens whose cumulative probability is at least $p$:

1. Sort tokens by probability (descending)
2. Compute cumulative probabilities
3. Keep tokens until cumulative probability $\geq p$
4. Remove all other tokens
5. Renormalize and sample

**Advantage over top-k:** The number of tokens varies depending on the distribution shape.
- Peaked distribution: only 1-2 tokens needed (focused)
- Flat distribution: many tokens included (diverse)

**Typical values:** $p \in [0.9, 0.95]$

---
## 7. Code: Implement Top-P Filtering

In [ ]:
def top_p_filtering(logits, p):
    """
    Filter logits to keep the smallest set of tokens with cumulative probability >= p.
    All other logits are set to -inf.
    
    Args:
        logits: (vocab_size,) tensor of raw logits
        p: cumulative probability threshold (0 < p <= 1)
    Returns:
        Filtered logits (vocab_size,) tensor
    """
    if p <= 0 or p > 1:
        raise ValueError("p must be in (0, 1]")
    if p == 1.0:
        return logits  # keep all
    
    # Sort logits in descending order
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    
    # Compute cumulative probabilities
    sorted_probs = F.softmax(sorted_logits, dim=-1)
    cumulative_probs = torch.cumsum(sorted_probs, dim=-1)
    
    # Find tokens to remove: cumulative prob exceeds p (shift right by 1 to keep the first token that crosses p)
    sorted_mask = cumulative_probs - sorted_probs >= p  # True for tokens to remove
    
    # Set removed logits to -inf
    sorted_logits[sorted_mask] = float('-inf')
    
    # Unsort back to original order
    filtered = torch.empty_like(logits)
    filtered.scatter_(0, sorted_indices, sorted_logits)
    
    return filtered


# Demonstrate
print("Original probabilities (T=1.0):")
probs_orig = F.softmax(logits, dim=-1)
for token, prob in zip(vocab, probs_orig.tolist()):
    print(f"  {token}: {prob:.4f}")

print()
for p in [0.5, 0.8, 0.9, 0.95]:
    filtered = top_p_filtering(logits, p)
    probs = F.softmax(filtered, dim=-1)
    active = [(token, prob.item()) for token, prob in zip(vocab, probs) if prob > 0]
    print(f"Top-p={p}: {len(active)} tokens kept")
    for token, prob in active:
        print(f"  {token}: {prob:.4f}")
    print()

In [ ]:
# Visualize top-p effects
p_values = [0.5, 0.8, 0.9, 0.99]

fig, axes = plt.subplots(1, len(p_values), figsize=(4 * len(p_values), 4))

for ax, p in zip(axes, p_values):
    filtered = top_p_filtering(logits, p)
    probs = F.softmax(filtered, dim=-1).numpy()
    
    colors = ['steelblue' if prob > 0 else 'lightgray' for prob in probs]
    ax.bar(range(len(vocab)), probs, color=colors, alpha=0.8)
    ax.set_xticks(range(len(vocab)))
    ax.set_xticklabels(vocab, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 0.8)
    ax.set_title(f'Top-P = {p}', fontsize=12)
    ax.set_ylabel('Probability')
    
    n_active = sum(1 for prob in probs if prob > 0)
    ax.text(0.95, 0.95, f'{n_active} tokens', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle('Effect of Top-P (Nucleus) on Token Selection', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 8. Combining Strategies

In practice, these strategies are often **combined**:

1. Apply **temperature** to logits
2. Apply **top-k** filtering
3. Apply **top-p** filtering
4. Sample from the resulting distribution

Common combinations:
- `temperature=0.7, top_p=0.9` -- balanced creative output
- `temperature=0.2, top_k=1` -- nearly deterministic
- `temperature=1.0, top_p=0.95` -- diverse but coherent

In [ ]:
def sample_token(logits, temperature=1.0, top_k=0, top_p=1.0):
    """
    Sample a token from logits with temperature, top-k, and top-p.
    
    Args:
        logits: (vocab_size,) tensor of raw logits
        temperature: Temperature for scaling (default 1.0)
        top_k: If > 0, keep only top-k tokens (default 0 = disabled)
        top_p: If < 1.0, apply nucleus sampling (default 1.0 = disabled)
    Returns:
        Sampled token index (int)
    """
    # Step 1: Temperature
    scaled_logits = logits / temperature
    
    # Step 2: Top-k filtering
    if top_k > 0:
        scaled_logits = top_k_filtering(scaled_logits, top_k)
    
    # Step 3: Top-p filtering
    if top_p < 1.0:
        scaled_logits = top_p_filtering(scaled_logits, top_p)
    
    # Step 4: Convert to probabilities and sample
    probs = F.softmax(scaled_logits, dim=-1)
    token_idx = torch.multinomial(probs, num_samples=1).item()
    
    return token_idx


# Demonstrate sampling with different configs
set_global_seed(42)

configs = [
    {'temperature': 1.0, 'top_k': 0, 'top_p': 1.0, 'label': 'Default (T=1, no filtering)'},
    {'temperature': 0.3, 'top_k': 0, 'top_p': 1.0, 'label': 'Low temp (T=0.3)'},
    {'temperature': 2.0, 'top_k': 0, 'top_p': 1.0, 'label': 'High temp (T=2.0)'},
    {'temperature': 1.0, 'top_k': 3, 'top_p': 1.0, 'label': 'Top-K=3'},
    {'temperature': 1.0, 'top_k': 0, 'top_p': 0.8, 'label': 'Top-P=0.8'},
    {'temperature': 0.7, 'top_k': 5, 'top_p': 0.9, 'label': 'T=0.7, K=5, P=0.9'},
]

N_SAMPLES = 1000

print(f"Sampling {N_SAMPLES} tokens from each configuration:\n")
for config in configs:
    label = config.pop('label')
    counts = Counter()
    for _ in range(N_SAMPLES):
        idx = sample_token(logits, **config)
        counts[vocab[idx]] += 1
    
    # Show distribution
    top_3 = counts.most_common(3)
    dist_str = ", ".join(f"{t}: {c/N_SAMPLES:.1%}" for t, c in top_3)
    unique = len(counts)
    print(f"{label:35s} | Unique: {unique:2d} | Top-3: {dist_str}")
    config['label'] = label  # restore

---
## 9. Code: Demonstrate Effects with Random Logits

Let us see how these strategies behave on different **logit shapes** -- peaked vs flat distributions.

In [ ]:
set_global_seed(42)

VOCAB_SIZE = 50
token_names = [f"tok_{i}" for i in range(VOCAB_SIZE)]

# Create three different logit distributions
# 1. Peaked: one token dominates
peaked_logits = torch.randn(VOCAB_SIZE) * 0.5
peaked_logits[0] = 5.0  # one strong candidate

# 2. Flat: many tokens are roughly equally likely
flat_logits = torch.randn(VOCAB_SIZE) * 0.3

# 3. Multi-modal: a few strong candidates
multi_logits = torch.randn(VOCAB_SIZE) * 0.5
multi_logits[0] = 3.0
multi_logits[5] = 2.8
multi_logits[10] = 2.5

distributions = [
    ('Peaked', peaked_logits),
    ('Flat', flat_logits),
    ('Multi-modal', multi_logits),
]

fig, axes = plt.subplots(3, 4, figsize=(20, 12))

settings = [
    ('Original (T=1)', {'temperature': 1.0}),
    ('T=0.5', {'temperature': 0.5}),
    ('Top-K=5', {'temperature': 1.0, 'top_k': 5}),
    ('Top-P=0.9', {'temperature': 1.0, 'top_p': 0.9}),
]

for row, (dist_name, dist_logits) in enumerate(distributions):
    for col, (setting_name, params) in enumerate(settings):
        ax = axes[row, col]
        
        # Apply transformations
        processed = dist_logits / params.get('temperature', 1.0)
        if 'top_k' in params and params['top_k'] > 0:
            processed = top_k_filtering(processed, params['top_k'])
        if 'top_p' in params and params['top_p'] < 1.0:
            processed = top_p_filtering(processed, params['top_p'])
        
        probs = F.softmax(processed, dim=-1).numpy()
        
        # Sort for visibility
        sorted_probs = np.sort(probs)[::-1]
        colors = ['steelblue' if p > 0.001 else 'lightgray' for p in sorted_probs]
        ax.bar(range(VOCAB_SIZE), sorted_probs, color=colors, alpha=0.8, width=1.0)
        
        n_active = sum(1 for p in probs if p > 0.001)
        entropy = -np.sum(probs * np.log(probs + 1e-10))
        
        if row == 0:
            ax.set_title(setting_name, fontsize=11)
        if col == 0:
            ax.set_ylabel(f'{dist_name}\nProbability', fontsize=10)
        
        ax.text(0.95, 0.95, f'Active: {n_active}\nH: {entropy:.2f}',
                transform=ax.transAxes, ha='right', va='top', fontsize=8,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
        ax.set_xlim(-1, VOCAB_SIZE)
        ax.set_ylim(0, max(sorted_probs) * 1.1 + 0.01)

plt.suptitle('Decoding Strategies on Different Logit Distributions\n(Tokens sorted by probability, descending)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Key observations:")
print("- Temperature affects all distributions similarly (sharper or flatter).")
print("- Top-k always keeps exactly k tokens, regardless of distribution shape.")
print("- Top-p adapts: keeps fewer tokens for peaked distributions, more for flat ones.")

In [ ]:
# Simulate autoregressive generation with different strategies
set_global_seed(42)

# Simple "language model": at each step, generate random logits and pick a token
# This illustrates the mechanics; a real model would condition on previous tokens

def simulate_generation(vocab, num_tokens, temperature=1.0, top_k=0, top_p=1.0, seed=42):
    """Simulate autoregressive generation with random logits."""
    set_global_seed(seed)
    generated = []
    for _ in range(num_tokens):
        # Random logits (simulating model output)
        logits = torch.randn(len(vocab))
        # Make some tokens more likely (simulate structured output)
        logits[0] += 1.0  # bias toward first few tokens
        logits[1] += 0.8
        logits[2] += 0.6
        
        idx = sample_token(logits, temperature=temperature, top_k=top_k, top_p=top_p)
        generated.append(vocab[idx])
    return generated


mini_vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran", "big",
              "red", "hat", "in", "to", "a", "is", "was"]

gen_configs = [
    {'temperature': 0.3, 'label': 'T=0.3 (confident)'},
    {'temperature': 1.0, 'label': 'T=1.0 (default)'},
    {'temperature': 2.0, 'label': 'T=2.0 (creative)'},
    {'temperature': 0.7, 'top_p': 0.9, 'label': 'T=0.7, P=0.9 (balanced)'},
    {'temperature': 1.0, 'top_k': 3, 'label': 'T=1.0, K=3 (restricted)'},
]

print("Simulated 20-token generations (random logits, not a real language model):\n")
for config in gen_configs:
    label = config.pop('label')
    tokens = simulate_generation(mini_vocab, 20, seed=42, **config)
    unique_ratio = len(set(tokens)) / len(tokens)
    print(f"{label:30s} | Unique: {unique_ratio:.0%} | {' '.join(tokens)}")
    config['label'] = label

---
## 10. Exercise: Experiment with Parameter Combinations

**Task:** Explore how different parameter combinations affect the output distribution.

1. Create a set of logits with 3 strong candidates and 47 weak ones
2. Apply the following combinations and compare the resulting distributions:
   - `T=0.5, K=10, P=1.0`
   - `T=1.0, K=0, P=0.85`
   - `T=0.7, K=10, P=0.9`
   - `T=1.5, K=5, P=0.95`
3. Sample 500 tokens from each and plot the token frequency distribution
4. Which combination gives the best balance of diversity and focus?

```python
# Your code here
#
# logits = torch.randn(50) * 0.5
# logits[0] = 3.0; logits[7] = 2.5; logits[15] = 2.0  # 3 strong candidates
#
# for config in configs:
#     # Sample 500 tokens, count frequencies, plot
#     ...
```

In [ ]:
# Try the exercise yourself before looking at the solution!





### Solution

In [ ]:
# ----- Solution -----
set_global_seed(42)

# Logits with 3 strong candidates
exercise_logits = torch.randn(50) * 0.5
exercise_logits[0] = 3.0
exercise_logits[7] = 2.5
exercise_logits[15] = 2.0

exercise_configs = [
    {'temperature': 0.5, 'top_k': 10, 'top_p': 1.0, 'label': 'T=0.5, K=10'},
    {'temperature': 1.0, 'top_k': 0,  'top_p': 0.85, 'label': 'T=1.0, P=0.85'},
    {'temperature': 0.7, 'top_k': 10, 'top_p': 0.9, 'label': 'T=0.7, K=10, P=0.9'},
    {'temperature': 1.5, 'top_k': 5,  'top_p': 0.95, 'label': 'T=1.5, K=5, P=0.95'},
]

N = 500
fig, axes = plt.subplots(1, len(exercise_configs), figsize=(5 * len(exercise_configs), 4))

for ax, config in zip(axes, exercise_configs):
    label = config.pop('label')
    counts = Counter()
    for _ in range(N):
        idx = sample_token(exercise_logits, **config)
        counts[idx] += 1
    config['label'] = label
    
    # Plot frequency distribution
    freq = np.zeros(50)
    for idx, count in counts.items():
        freq[idx] = count / N
    
    sorted_freq = np.sort(freq)[::-1]
    ax.bar(range(50), sorted_freq, color='steelblue', alpha=0.8, width=1.0)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Token rank')
    ax.set_ylabel('Frequency')
    
    unique = len(counts)
    top3_mass = sum(c for c in sorted(counts.values(), reverse=True)[:3]) / N
    ax.text(0.95, 0.95, f'Unique: {unique}\nTop-3: {top3_mass:.0%}',
            transform=ax.transAxes, ha='right', va='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.suptitle(f'Token Frequency from {N} Samples per Configuration', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("Observation: T=0.7, K=10, P=0.9 often gives a good balance --")
print("the 3 strong candidates get most probability mass, but there is still some diversity.")

---
## 11. Common Mistakes & Debugging Tips

**1. Setting temperature to 0**
- Temperature must be strictly positive; $T=0$ causes division by zero
- For greedy decoding, use `argmax` directly instead of very low temperature

**2. Applying top-k/top-p to probabilities instead of logits**
- Filter on logits (set removed entries to $-\infty$), then apply softmax
- Filtering on probabilities and renormalizing can give slightly different results

**3. Top-p cumulative sum off-by-one**
- The token that crosses the cumulative threshold should be **included**, not excluded
- Check: after filtering, the sum of remaining probabilities should be $\geq p$

**4. Not renormalizing after filtering**
- After top-k or top-p, probabilities no longer sum to 1
- Applying softmax to the filtered logits handles renormalization automatically

**5. Confusing logits and probabilities**
- Logits are raw model outputs (can be any real number)
- Probabilities are after softmax (sum to 1, all non-negative)
- Temperature divides **logits**, not probabilities

**6. Top-k with k=1 is greedy, not sampling**
- `top_k=1` always picks the single most probable token
- There is no randomness; every call returns the same token

**7. Forgetting that strategies interact**
- High temperature + low top-k: temperature spreads mass, then top-k cuts it
- Low temperature + high top-p: temperature concentrates mass, so top-p keeps few tokens
- Always think about the order: temperature first, then filtering

---

**Next notebook:** [07 - RAG Concept Only (Optional)](07_RAG_Concept_Only_optional.ipynb) -- learn the Retrieval-Augmented Generation (RAG) pattern for grounding LLM outputs in retrieved documents.